In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
#!/usr/bin/env python3
# ============================================================
# Con φ  ~  feature pipeline
# ============================================================
#
# VERSION BLOCK
# -------------
# Set these two variables once here. All paths and output
# filenames are derived from them automatically.
# If you are running this notebook standalone (not via the
# master runner), uncomment the lines below and set your own
# BASE_DIR.
#
VERSION  = "v1"
BASE_DIR_DEFAULT = "/content/drive/MyDrive/conphi"
#
# ── Per-script override (uncomment if running standalone) ────
# from pathlib import Path
# VERSION  = "v1"
# BASE_DIR = Path("/content/drive/MyDrive/conphi")   # 👈 change to your path
# ─────────────────────────────────────────────────────────────
#
# If BASE_DIR is not already defined (i.e. running standalone
# without the master runner), fall back to the default above.
try:
    BASE_DIR
except NameError:
    from pathlib import Path
    BASE_DIR = Path(BASE_DIR_DEFAULT)

BASE_DIR = Path(BASE_DIR)

# ============================================================
# PURPOSE
# -------
# Build vintage-controlled model input files that feed both
# Con φ sub-models:
#   - Con φ USE  (Update Survey Estimate)
#   - Con φ WASE (Without Any Survey Estimate)
#
# For each vintage year t, produces:
#   conphi_{VERSION}_features_{t}.parquet
# containing ONLY information that was available as of year t.
# This is the foundational leakage-prevention mechanism for the
# entire Con φ system.
#
# VINTAGE CONTROL ARCHITECTURE
# ----------------------------
# The pipeline produces one parquet file per vintage year. Each
# file contains a full iso × year × percentile panel, but with
# strict controls on what information is included:
#
#   IMF macro data:    loaded from the WEO vintage published in
#                      October of year t (file named {t}_10.csv).
#                      GDP forecasts beyond t are WEO projections,
#                      not realised values.
#
#   Survey outcomes:   consumption percentiles from PIP. These
#                      are NOT masked here — masking happens in
#                      the model scripts' train/test splitting.
#                      The pipeline provides all available data;
#                      the models enforce temporal discipline.
#
#   Structural covs:   rural share, education, mortality, etc.
#                      are HELD CONSTANT after the vintage year
#                      (frozen at their last observed value ≤ t).
#                      This prevents future covariate values from
#                      leaking into predictions.
#
#   Lagged covariates: u5mort, rural_pct, edu_mean_fem, remit,
#                      res_rents, gov_rev, gov_exp are lagged
#                      3 years AND frozen after the vintage year.
#                      The lag prevents simultaneity bias; the
#                      freeze prevents look-ahead.
#
# GDP GROWTH DERIVATION
# ---------------------
# gdp_growth is computed as diff(log_gdp_pp), i.e. the first
# difference of log GDP per capita in PPP terms. This is
# preferred over pct_change(gdp_percap_lc_real) because:
#   1. log_gdp_pp (from PPPPC) has ~122 country coverage vs
#      ~85 for gdp_percap_lc_real (from NGDPRPC).
#   2. Summing diff(log) is exact: Σ diff(log) = log(GDP_T/GDP_0).
#   3. PPP units match the consumption target (2017 PPP USD/day).
#   4. Consistent with Con φ's use of log_gdp_pp as the level
#      covariate — growth is mechanically the first difference.
#
# GDP-CONDITIONED CASCADE IMPUTATION
# -----------------------------------
# gov_rev and gov_exp use a GDP-quartile-conditioned cascade:
#   region23×gdp_bin×year → region×gdp_bin×year → gdp_bin×year
#   → year → global median.
# This prevents high-income country values from contaminating
# LIC fills (and vice versa). GDP bins are anchored at the
# vintage year to prevent look-ahead.
#
# IMPUTATION STRATEGY
# -------------------
# Missing values are filled using the mildest defensible method:
#
#   remit:       leading zeros → interpolate → trailing ffill
#                (zero remittances before first observation is
#                plausible for many countries)
#
#   res_rents:   leading bfill → interpolate → trailing ffill
#                (zero resource rents is NOT plausible — a country
#                with oil in 2010 likely had oil in 2005)
#
#   gov_rev:     bfill → interpolate → ffill within country,
#                then GDP-conditioned cascade:
#                  region23×gdp_bin×year median
#                  → region×gdp_bin×year median
#                  → gdp_bin×year median
#                  → year median → global median.
#
#   gov_exp:     same as gov_rev.
#
# All imputation happens WITHIN the vintage-specific pipeline,
# so each vintage file is self-consistent.
#
# OUTPUT
# ------
# One parquet per vintage year in data - model inputs/:
#   conphi_v1_features_2015.parquet
#   conphi_v1_features_2016.parquet
#   ...
#   conphi_v1_features_2025.parquet
#
# Plus an audit block (Block 4) that runs coverage diagnostics
# on the most recent vintage, including outcome-conditional
# missingness (which covariates are missing for iso-years that
# DO have observed consumption).
#
# ============================================================

import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
np.random.seed(42)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

# ============================================================
# PATHS  (derived from BASE_DIR and VERSION — do not hardcode)
# ============================================================
RAW_DIR  = BASE_DIR / "data - raw"
IMF_DIR  = RAW_DIR / "imf"
WB_DIR   = RAW_DIR / "worldbank"
IHME_DIR = RAW_DIR / "ihme"
RAIN_DIR = RAW_DIR / "rainfall"

OUT_DIR   = BASE_DIR / "data - model inputs"
AUDIT_DIR = OUT_DIR / "audit"
OUT_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

# ── Output file naming ───────────────────────────────────────
# conphi_v1_features_{vintage_year}.parquet
OUTPUT_PREFIX = f"conphi_{VERSION}_features"

# ============================================================
# CONFIG
# ============================================================

# ── Source file names ────────────────────────────────────────
PIPS_PERCENTILES_FILE  = "wb_pips_pctiles_2025.csv"
PIPS_WIDE_FILE         = "wb_pips_wide_2025.csv"
REGION_INFO_FILE       = "region_info.csv"
EXCHANGE_RATE_FILE     = "wb_exchange_rate.csv"
RURAL_URBAN_FILE       = "wb_rural_urban.csv"
IHME_EDU_FILE          = "ihme_edu.csv"
RAINFALL_FILE          = "rainfall_filled.csv"
WB_LIFEEXP_FILE        = "wb_lifeexp.csv"
WB_U5MORT_FILE         = "wb_u5mort.csv"
WB_REMIT_FILE          = "wb_remit.csv"
WB_RESOURCE_RENTS_FILE = "wb_resource_rents.csv"

# ── Build scope ──────────────────────────────────────────────
RUN_MODE   = "multi"
START_YEAR = 2015
END_YEAR   = 2025
# TEST_YEAR = 2018   # uncomment for single-vintage runs

AUDIT_VINTAGE = 2025
DROP_YEARS    = []

# ── Kosovo handling ──────────────────────────────────────────
KOSOVO_ALIASES    = {"UVK": "XKX", "KOS": "XKX"}
KOSOVO_TARGET_ISO = "XKX"
KOSOVO_PROXY_ISO  = "SRB"

DROP_ISO = []

# ── PIP settings ─────────────────────────────────────────────
MAX_PERCENTILE    = 99
PIP_PCT_HARD_DROP = ["avg_welfare", "welfare_share", "pop_share"]

# ── IMF settings ─────────────────────────────────────────────
IMF_YEAR_MIN           = 1970
IMF_YEAR_MAX           = 2030
FORECAST_HORIZON_YEARS = 5

# ── Palestine special handling ───────────────────────────────
PSE_DONOR_VINTAGE    = 2014
PSE_DONOR_SOURCE_ISO = "WBG"
PSE_TARGET_ISO       = "PSE"

# ── IMF WEO code → human-readable name mapping ──────────────
IMF_CODE_MAP_DIRECT = {
    "PCPIPCH":     "inflation",
    "BCA_NGDPD":   "current_account_gdp",
    "GGR_NGDP":    "gov_rev",
    "GGX_NGDP":    "gov_exp",
    "GGXWDG_NGDP": "debt_gdp",
    "PCPI":        "cpi",
    "PPPPC":       "gdp_percap_ppp",
    "NGDPPC":      "gdp_percap_lc_current",
    "NGDPRPC":     "gdp_percap_lc_real",
    "GGXCNL_NGDP": "fiscal_balance",
    "NID_NGDP":    "investments_gdp",
}

# ── Region hierarchy ─────────────────────────────────────────
REGION_PARENT_FROM_R23 = {
    "Eastern Africa":   "Sub-Saharan Africa",
    "Middle Africa":    "Sub-Saharan Africa",
    "Southern Africa":  "Sub-Saharan Africa",
    "Western Africa":   "Sub-Saharan Africa",
    "Northern Africa":  "Middle East & North Africa",
    "Western Asia":     "Middle East & North Africa",
    "Southern Asia":    "South Asia",
    "Central Asia":     "Europe & Central Asia",
    "Eastern Europe":   "Europe & Central Asia",
    "Northern Europe":  "Europe & Central Asia",
    "Southern Europe":  "Europe & Central Asia",
    "Eastern Asia":     "East Asia & Pacific",
    "South-Eastern Asia": "East Asia & Pacific",
    "Polynesia":        "East Asia & Pacific",
    "Melanesia":        "East Asia & Pacific",
    "Micronesia":       "East Asia & Pacific",
    "Caribbean":        "Latin America & Caribbean",
    "Central America":  "Latin America & Caribbean",
    "South America":    "Latin America & Caribbean",
}

REGION23_OVERRIDE_BY_ISO = {
    "SDN": "Eastern Africa",
    "SSD": "Eastern Africa",
    "XKX": "Southern Europe",
}

# ── Post-vintage freeze columns ──────────────────────────────
POST_VINTAGE_HOLD_CONSTANT_COLS = [
    "exchange_rt",
    "rural_pct",
    "edu_mean_fem",
    "edu_mean_male",
    "rainfall_anom",
    "lifeexp",
    "u5mort",
    "remit",
    "res_rents",
]

# ── Lag settings ─────────────────────────────────────────────
LAG_YEARS = 3

# ── GDP bin settings for cascade imputation ──────────────────
N_GDP_BINS    = 4
GDP_BIN_LABELS = [f"Q{i+1}" for i in range(N_GDP_BINS)]


# ============================================================
# HELPERS
# ============================================================
def _strip_colnames(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df


def _norm_iso(series, kosovo_aliases=KOSOVO_ALIASES):
    s = series.astype(str).str.strip().str.upper()
    if kosovo_aliases:
        s = s.replace(kosovo_aliases)
    return s


def _to_float(s):
    return pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)


def _drop_bad_iso_codes(df, iso_col="iso"):
    df = df.copy()
    if iso_col not in df.columns:
        return df
    df[iso_col] = _norm_iso(df[iso_col])
    df = df[df[iso_col].astype(str).str.fullmatch(r"[A-Z]{3}", na=False)].copy()
    if DROP_ISO:
        df = df[~df[iso_col].isin(DROP_ISO)].copy()
    return df


def _add_kosovo_proxy_from_srb(df, iso_col="iso"):
    df = df.copy()
    if iso_col not in df.columns:
        return df
    df[iso_col] = _norm_iso(df[iso_col])
    has_xkx = (df[iso_col] == KOSOVO_TARGET_ISO).any()
    has_srb = (df[iso_col] == KOSOVO_PROXY_ISO).any()
    if (not has_xkx) and has_srb:
        proxy = df[df[iso_col] == KOSOVO_PROXY_ISO].copy()
        proxy[iso_col] = KOSOVO_TARGET_ISO
        df = pd.concat([df, proxy], ignore_index=True)
    return df


def merge_safe(left, right, on, how="left", prefer_right=None):
    prefer_right = set(prefer_right or [])
    on = list(on)
    L = left.copy(); R = right.copy()
    overlap = (set(L.columns) & set(R.columns)) - set(on)
    drop_from_right = [c for c in overlap if c not in prefer_right]
    if drop_from_right:
        R = R.drop(columns=drop_from_right, errors="ignore")
    out = L.merge(R, on=on, how=how)
    bad = [c for c in out.columns if c.endswith("_x") or c.endswith("_y")]
    if bad:
        raise RuntimeError(f"Unexpected suffix columns after merge: {bad[:20]}")
    return out


# ── Imputation helpers ───────────────────────────────────────

def _fill_leading_zero_interpolate_ffill(series):
    """Leading zeros → interpolate → trailing ffill."""
    s = series.copy()
    first_valid = s.first_valid_index()
    if first_valid is None:
        return s.fillna(0.0)
    s.loc[:first_valid] = s.loc[:first_valid].fillna(0.0)
    s = s.interpolate(method="linear")
    s = s.ffill()
    return s


def impute_leading_zero_interp_ffill(df, cols):
    df = df.sort_values(["iso", "year"]).copy()
    for c in cols:
        if c in df.columns:
            df[c] = _to_float(df[c])
            df[c] = df.groupby("iso")[c].transform(_fill_leading_zero_interpolate_ffill)
    return df


def _fill_bfill_interpolate_ffill(series):
    """bfill → interpolate → ffill."""
    s = series.copy()
    first_valid = s.first_valid_index()
    if first_valid is None:
        return s
    s = s.interpolate(method="linear")
    s = s.ffill()
    s = s.bfill()
    return s


def impute_bfill_interp_ffill(df, cols):
    df = df.sort_values(["iso", "year"]).copy()
    for c in cols:
        if c in df.columns:
            df[c] = _to_float(df[c])
            df[c] = df.groupby("iso")[c].transform(_fill_bfill_interpolate_ffill)
    return df


# ── GDP-conditioned cascade imputation ───────────────────────

def cascade_impute_by_gdp_bin(df, col, vintage_year):
    """Impute `col` via bfill/interp/ffill within country, then
    cascade through region23×gdp_bin×year, region×gdp_bin×year,
    gdp_bin×year, year, and global medians.

    GDP bins are computed from log_gdp_pp AT THE VINTAGE YEAR
    to prevent look-ahead. Countries missing GDP entirely get
    a fallback bin based on available years ≤ vintage_year.
    """
    df[col] = _to_float(df[col])
    df = impute_bfill_interp_ffill(df, cols=[col])

    # Build GDP bins anchored at vintage year
    gdp_anchor = (
        df.loc[df["year"] <= vintage_year, ["iso", "year", "log_gdp_pp"]]
        .dropna(subset=["log_gdp_pp"])
        .sort_values(["iso", "year"])
        .groupby("iso")["log_gdp_pp"]
        .last()
    )

    if len(gdp_anchor) >= N_GDP_BINS:
        bin_edges = pd.qcut(
            gdp_anchor, q=N_GDP_BINS, labels=GDP_BIN_LABELS, duplicates="drop"
        )
        iso_to_gdp_bin = bin_edges.to_dict()
    else:
        iso_to_gdp_bin = {}

    df["_gdp_bin"] = df["iso"].map(iso_to_gdp_bin)

    # Cascade: region23×gdp_bin×year → region×gdp_bin×year
    #        → gdp_bin×year → year → global
    cascade_levels = [
        ["region23", "_gdp_bin", "year"],
        ["region",   "_gdp_bin", "year"],
        ["_gdp_bin", "year"],
        ["year"],
    ]
    for levels in cascade_levels:
        valid = [l for l in levels if l in df.columns]
        if not valid:
            continue
        fill = df.groupby(valid, observed=True)[col].transform("median")
        df[col] = df[col].fillna(fill)

    # Final global fallback
    df[col] = df[col].fillna(df[col].median())
    df = df.drop(columns=["_gdp_bin"])

    n_still_missing = df[col].isna().sum()
    if n_still_missing > 0:
        print(f"  ⚠ {col}: {n_still_missing:,} rows still missing after GDP-conditioned cascade")

    return df


# ── Lag and freeze helpers ───────────────────────────────────

def add_lags(df, cols, lag=3):
    df = df.sort_values(["iso", "year"]).copy()
    for c in cols:
        if c in df.columns:
            df[f"{c}_lag{lag}"] = df.groupby("iso")[c].shift(lag)
    return df


def add_lags_and_freeze_after_vintage(df, cols, lag=3, vintage_year=None):
    """Create lagged covariates AND freeze them after vintage_year."""
    df = df.sort_values(["iso", "year"]).copy()
    df = add_lags(df, cols=cols, lag=lag)
    if vintage_year is None:
        return df
    for c in cols:
        lag_col = f"{c}_lag{lag}"
        if lag_col not in df.columns:
            continue
        def _freeze_group(g):
            g = g.sort_values("year").copy()
            anchor_rows = g.loc[g["year"] == vintage_year, lag_col].dropna()
            if len(anchor_rows) == 0:
                anchor_rows = g.loc[g["year"] <= vintage_year, lag_col].dropna()
            if len(anchor_rows) == 0:
                return g[lag_col]
            anchor_value = anchor_rows.iloc[-1]
            g.loc[g["year"] > vintage_year, lag_col] = anchor_value
            return g[lag_col]
        df[lag_col] = df.groupby("iso", group_keys=False).apply(_freeze_group)
    return df


def hold_constant_after_vintage(df, cols, vintage_year):
    """Freeze non-IMF covariates at their last value ≤ vintage_year."""
    df = df.sort_values(["iso", "year"]).copy()
    for c in cols:
        if c not in df.columns:
            continue
        def _fill_group(g):
            g = g.sort_values("year").copy()
            anchor = g.loc[g["year"] <= vintage_year, c].dropna()
            if len(anchor) == 0:
                return g[c]
            anchor_value = anchor.iloc[-1]
            g.loc[g["year"] > vintage_year, c] = anchor_value
            return g[c]
        df[c] = df.groupby("iso", group_keys=False).apply(_fill_group)
    return df


def enforce_nested_regions(df):
    """Ensure region23 → region mapping is clean and exhaustive."""
    df = df.copy()
    if "region23" not in df.columns:
        df["region23"] = np.nan
    if "region" not in df.columns:
        df["region"] = np.nan
    df["region23"] = df["iso"].map(REGION23_OVERRIDE_BY_ISO).fillna(df["region23"])
    known_r23 = df["region23"].isin(REGION_PARENT_FROM_R23.keys())
    df.loc[known_r23, "region"] = df.loc[known_r23, "region23"].map(REGION_PARENT_FROM_R23)
    missing_r23 = sorted(
        set(df["region23"].dropna().unique()) - set(REGION_PARENT_FROM_R23.keys())
    )
    if missing_r23:
        raise ValueError(f"Missing parent mapping for region23 values: {missing_r23}")
    chk = (
        df[["region23", "region"]].dropna().drop_duplicates()
        .groupby("region23")["region"].nunique()
    )
    bad = chk[chk > 1]
    if len(bad) > 0:
        raise ValueError(f"region23 not cleanly nested in region for: {bad.index.tolist()}")
    return df


def build_iso_year_panel(keep_isos, min_year, max_year):
    return pd.MultiIndex.from_product(
        [sorted(keep_isos), list(range(min_year, max_year + 1))],
        names=["iso", "year"]
    ).to_frame(index=False)


def expand_outcomes_to_panel(panel, pip_pct):
    return merge_safe(panel, pip_pct, on=["iso", "year"], how="left")


# ============================================================
# IMF STANDARDISATION
# ============================================================
def _standardize_imf_raw(fp):
    raw = _strip_colnames(pd.read_csv(fp))
    col_lut = {c.lower(): c for c in raw.columns}
    iso_col = col_lut.get("iso", None)
    weo_col = col_lut.get("weo subject code", None)
    if iso_col is None or weo_col is None:
        raise KeyError(f"Could not find IMF iso / weo code columns in {fp.name}")
    raw = raw.rename(columns={iso_col: "iso", weo_col: "weo_code"})
    raw["iso"] = _norm_iso(raw["iso"])
    raw["iso"] = raw["iso"].replace({PSE_DONOR_SOURCE_ISO: PSE_TARGET_ISO})
    year_cols = [c for c in raw.columns if str(c).strip().isdigit()]
    return raw[["iso", "weo_code"] + year_cols].copy()


def _inject_pse_from_2014(raw_current, vintage_year, donor_2014):
    if vintage_year >= PSE_DONOR_VINTAGE:
        return raw_current
    if donor_2014 is None or donor_2014.empty:
        return raw_current
    current_year_cols = [c for c in raw_current.columns if str(c).isdigit()]
    if not current_year_cols:
        return raw_current
    donor = donor_2014[donor_2014["iso"] == PSE_TARGET_ISO].copy()
    if donor.empty:
        return raw_current
    donor_year_cols = [c for c in donor.columns if str(c).isdigit()]
    use_year_cols   = [c for c in current_year_cols if c in donor_year_cols]
    donor = donor[["iso", "weo_code"] + use_year_cols].copy()
    for yc in current_year_cols:
        if yc not in donor.columns:
            donor[yc] = np.nan
    donor   = donor[["iso", "weo_code"] + current_year_cols].copy()
    current = raw_current[raw_current["iso"] != PSE_TARGET_ISO].copy()
    return pd.concat([current, donor], ignore_index=True)


# ============================================================
# LOADERS
# ============================================================
def load_imf_all():
    files = sorted(IMF_DIR.glob("*_10.csv"))
    if not files:
        raise FileNotFoundError(f"No IMF '*_10.csv' files in {IMF_DIR}")

    donor_fp = None
    for fp in files:
        m = re.search(r"(19|20)\d{2}", fp.name)
        if m and int(m.group(0)) == PSE_DONOR_VINTAGE:
            donor_fp = fp; break
    donor_2014 = _standardize_imf_raw(donor_fp) if donor_fp else None

    out = {}
    for fp in files:
        m = re.search(r"(19|20)\d{2}", fp.name)
        if not m:
            continue
        vintage_year = int(m.group(0))
        raw = _standardize_imf_raw(fp)
        raw = _inject_pse_from_2014(raw, vintage_year, donor_2014)
        year_cols = [c for c in raw.columns if str(c).strip().isdigit()]
        long = raw[["iso", "weo_code"] + year_cols].melt(
            id_vars=["iso", "weo_code"], value_vars=year_cols,
            var_name="year", value_name="value",
        )
        long["year"]  = pd.to_numeric(long["year"], errors="coerce")
        long = long.dropna(subset=["year"]).copy()
        long["year"]  = long["year"].astype(int)
        long["value"] = pd.to_numeric(
            long["value"].astype(str).str.replace(",", "", regex=False), errors="coerce",
        )
        long = long[(long["year"] >= IMF_YEAR_MIN) & (long["year"] <= IMF_YEAR_MAX)].copy()
        long["weo_code"] = long["weo_code"].astype(str).str.strip()
        long = long[~long["weo_code"].isin(["nan", "None", ""])]
        if DROP_YEARS:
            long = long[~long["year"].isin(DROP_YEARS)].copy()
        long = _drop_bad_iso_codes(long, "iso")
        long = _add_kosovo_proxy_from_srb(long, "iso")
        out[vintage_year] = long

    print(f"✓ IMF vintages loaded: {sorted(out.keys())}")
    return out


def build_imf_wide_for_vintage(imf_dict, base_vintage):
    if int(base_vintage) not in imf_dict:
        raise ValueError(f"Vintage {base_vintage}_10.csv not found")
    df = imf_dict[int(base_vintage)].copy()
    wide = (
        df.pivot_table(index=["iso", "year"], columns="weo_code",
                       values="value", aggfunc="first")
          .reset_index()
    )
    wide.columns.name = None
    raw_imf_cols = [c for c in wide.columns if c not in ["iso", "year"]]
    wide = wide.rename(columns=IMF_CODE_MAP_DIRECT)

    if "gdp_percap_ppp" in wide.columns:
        wide["log_gdp_pp"] = np.where(
            wide["gdp_percap_ppp"] > 0, np.log(wide["gdp_percap_ppp"]), np.nan
        )
    else:
        wide["log_gdp_pp"] = np.nan

    wide = wide.sort_values(["iso", "year"]).copy()
    wide["gdp_growth"] = wide.groupby("iso")["log_gdp_pp"].diff()

    renamed_cols = set(IMF_CODE_MAP_DIRECT.values())
    keep  = ["iso", "year", "gdp_growth", "log_gdp_pp"]
    keep += [c for c in wide.columns if c in renamed_cols]
    keep += [c for c in raw_imf_cols if c in wide.columns and c not in keep]
    keep  = list(dict.fromkeys(keep))
    wide  = wide[keep].copy()
    for c in wide.columns:
        if c not in ["iso", "year"]:
            wide[c] = _to_float(wide[c])
    wide = _drop_bad_iso_codes(wide, "iso")
    wide = _add_kosovo_proxy_from_srb(wide, "iso")
    return wide.sort_values(["iso", "year"]).reset_index(drop=True)


def load_pip_percentiles():
    p   = WB_DIR / PIPS_PERCENTILES_FILE
    pct = pd.read_csv(p)
    pct = pct.rename(columns={"country_code": "iso", "quantile": "cons"})
    pct = pct.drop(columns=[c for c in PIP_PCT_HARD_DROP if c in pct.columns], errors="ignore")
    pct["iso"]  = _norm_iso(pct["iso"])
    pct = pct[~((pct["iso"] == "SSD") & (pd.to_numeric(pct["year"], errors="coerce") == 2009))].copy()
    pct = pct[pct["reporting_level"].astype(str).str.lower() == "national"].copy()
    pct = pct[pct["percentile"] <= MAX_PERCENTILE].copy()
    pct["year"] = pd.to_numeric(pct["year"], errors="coerce")
    pct = pct.dropna(subset=["year"]).copy()
    pct["year"] = pct["year"].astype(int)
    pct["cons"] = _to_float(pct["cons"])
    if DROP_ISO:   pct = pct[~pct["iso"].isin(DROP_ISO)].copy()
    if DROP_YEARS: pct = pct[~pct["year"].isin(DROP_YEARS)].copy()
    pct = _drop_bad_iso_codes(pct, "iso")
    pct = pct[pct["welfare_type"] == "consumption"].copy()
    keep = ["iso", "year", "percentile", "cons", "welfare_type", "reporting_level"]
    if "pop" in pct.columns:
        pct["pop"] = _to_float(pct["pop"]); keep.append("pop")
    pct = (
        pct[keep]
        .sort_values(["iso", "year", "percentile"])
        .drop_duplicates(["iso", "year", "percentile"])
    )
    print(f"✓ PIP percentiles: {len(pct):,} rows")
    return pct


def load_pip_wide_comparability():
    p = WB_DIR / PIPS_WIDE_FILE
    if not p.exists():
        return pd.DataFrame(columns=["iso", "year", "survey_comparability", "comparable_spell_raw"])
    w = _strip_colnames(pd.read_csv(p))
    col_lut = {c.lower(): c for c in w.columns}
    renames  = {}
    for tgt, alts in [
        ("iso",   ["country_code", "iso3", "iso3c", "code"]),
        ("year",  ["reporting_year", "survey_year", "time"]),
        ("survey_comparability", ["survey_comparability"]),
        ("comparable_spell",     ["comparable_spell"]),
    ]:
        if tgt in col_lut:
            renames[col_lut[tgt]] = tgt
        else:
            for a in alts:
                if a in col_lut:
                    renames[col_lut[a]] = tgt; break
    w = w.rename(columns=renames)
    if "iso" not in w.columns or "year" not in w.columns:
        raise KeyError("PIP wide needs iso and year")
    if "reporting_level" in w.columns:
        w = w[w["reporting_level"].astype(str).str.lower().str.strip() == "national"].copy()
    w["iso"]  = _norm_iso(w["iso"])
    w = w[~((w["iso"] == "SSD") & (pd.to_numeric(w["year"], errors="coerce") == 2009))].copy()
    w["year"] = pd.to_numeric(w["year"], errors="coerce")
    w = w.dropna(subset=["year"]).copy()
    w["year"] = w["year"].astype(int)
    if DROP_ISO:   w = w[~w["iso"].isin(DROP_ISO)].copy()
    if DROP_YEARS: w = w[~w["year"].isin(DROP_YEARS)].copy()
    w = _drop_bad_iso_codes(w, "iso")
    for c in ["survey_comparability", "comparable_spell"]:
        if c not in w.columns:
            w[c] = np.nan
    out = (
        w[["iso", "year", "survey_comparability", "comparable_spell"]]
        .sort_values(["iso", "year"])
        .groupby(["iso", "year"], as_index=False).first()
    )
    out["survey_comparability"] = _to_float(out["survey_comparability"])
    out = out.rename(columns={"comparable_spell": "comparable_spell_raw"})
    print(f"✓ PIP comparability: {len(out):,} rows")
    return out


def load_exchange():
    p   = WB_DIR / EXCHANGE_RATE_FILE
    raw = _strip_colnames(pd.read_csv(p))
    raw.columns = (
        raw.columns
        .str.replace(r"\s*\[YR\d+\]", "", regex=True)
        .str.lower().str.replace(" ", "_")
    )
    year_cols = [c for c in raw.columns if c.isdigit()]
    long = raw.melt(
        id_vars="country_code", value_vars=year_cols,
        var_name="year", value_name="exchange_rt",
    )
    long["iso"]         = _norm_iso(long["country_code"])
    long["year"]        = pd.to_numeric(long["year"], errors="coerce")
    long = long.dropna(subset=["year"]).copy()
    long["year"]        = long["year"].astype(int)
    long["exchange_rt"] = _to_float(long["exchange_rt"])
    if DROP_ISO:   long = long[~long["iso"].isin(DROP_ISO)].copy()
    if DROP_YEARS: long = long[~long["year"].isin(DROP_YEARS)].copy()
    long = _drop_bad_iso_codes(long, "iso")
    long = _add_kosovo_proxy_from_srb(long, "iso")
    long = long[["iso", "year", "exchange_rt"]].drop_duplicates()
    print(f"✓ Exchange rate: {len(long):,} rows")
    return long


def load_rurality():
    p  = WB_DIR / RURAL_URBAN_FILE
    ru = _strip_colnames(pd.read_csv(p))
    ru.columns = ru.columns.str.lower().str.replace(" ", "_")
    year_cols  = [c for c in ru.columns if c.isdigit()]
    long = ru.melt(
        id_vars="country_code", value_vars=year_cols,
        var_name="year", value_name="rural_pct",
    )
    long["iso"]       = _norm_iso(long["country_code"])
    long["year"]      = pd.to_numeric(long["year"], errors="coerce")
    long = long.dropna(subset=["year"]).copy()
    long["year"]      = long["year"].astype(int)
    long["rural_pct"] = _to_float(long["rural_pct"])
    if DROP_ISO:   long = long[~long["iso"].isin(DROP_ISO)].copy()
    if DROP_YEARS: long = long[~long["year"].isin(DROP_YEARS)].copy()
    long = _drop_bad_iso_codes(long, "iso")
    long = _add_kosovo_proxy_from_srb(long, "iso")
    long = long[["iso", "year", "rural_pct"]].drop_duplicates()
    print(f"✓ Rural share: {len(long):,} rows")
    return long


def load_regions():
    p   = WB_DIR / REGION_INFO_FILE
    r   = _strip_colnames(pd.read_csv(p))
    r["iso3c"] = _norm_iso(r["iso3c"])
    out = (
        r[["iso3c", "region", "region23"]]
        .rename(columns={"iso3c": "iso"})
        .drop_duplicates("iso")
    )
    out = _drop_bad_iso_codes(out, "iso")
    out = _add_kosovo_proxy_from_srb(out, "iso")
    print(f"✓ Regions: {len(out)} ISOs")
    return out


def load_ihme_edu():
    p   = IHME_DIR / IHME_EDU_FILE
    edu = pd.read_csv(p)
    edu["iso"] = _norm_iso(edu["iso"])
    edu = _drop_bad_iso_codes(edu, "iso")
    edu = _add_kosovo_proxy_from_srb(edu, "iso")
    edu["year"] = pd.to_numeric(edu["year"], errors="coerce")
    edu = edu.dropna(subset=["year"]).copy()
    edu["year"] = edu["year"].astype(int)
    piv = edu.pivot_table(index=["iso", "year"], columns="sex", values="mean").reset_index()
    piv.columns.name = None
    piv = piv.rename(columns={"Female": "edu_mean_fem", "Male": "edu_mean_male"})
    for c in ["edu_mean_fem", "edu_mean_male"]:
        if c in piv.columns:
            piv[c] = _to_float(piv[c]).round(1)
    if DROP_ISO:   piv = piv[~piv["iso"].isin(DROP_ISO)].copy()
    if DROP_YEARS: piv = piv[~piv["year"].isin(DROP_YEARS)].copy()
    piv = _drop_bad_iso_codes(piv, "iso")
    piv = _add_kosovo_proxy_from_srb(piv, "iso")
    print(f"✓ IHME education: {len(piv):,} rows")
    return piv


def _load_wb_indicator_wide(filepath, value_name):
    if not filepath.exists():
        return pd.DataFrame(columns=["iso", "year", value_name])
    raw = _strip_colnames(pd.read_csv(filepath))
    raw.columns = (
        raw.columns
        .str.replace(r"\s*\[YR\d+\]", "", regex=True)
        .str.lower().str.replace(" ", "_")
    )
    iso_col = None
    for c in ["country_code", "countrycode", "iso", "iso3"]:
        if c in raw.columns: iso_col = c; break
    if iso_col is None:
        return pd.DataFrame(columns=["iso", "year", value_name])
    year_cols = [c for c in raw.columns if c.isdigit()]
    long = raw.melt(
        id_vars=iso_col, value_vars=year_cols,
        var_name="year", value_name=value_name,
    )
    long["iso"]      = _norm_iso(long[iso_col])
    long["year"]     = pd.to_numeric(long["year"], errors="coerce")
    long = long.dropna(subset=["year"]).copy()
    long["year"]     = long["year"].astype(int)
    long[value_name] = _to_float(long[value_name])
    if DROP_ISO:   long = long[~long["iso"].isin(DROP_ISO)].copy()
    if DROP_YEARS: long = long[~long["year"].isin(DROP_YEARS)].copy()
    long = _drop_bad_iso_codes(long, "iso")
    long = _add_kosovo_proxy_from_srb(long, "iso")
    long = long[["iso", "year", value_name]].drop_duplicates()
    print(f"✓ {value_name}: {len(long):,} rows")
    return long


def load_lifeexp():        return _load_wb_indicator_wide(WB_DIR / WB_LIFEEXP_FILE,        "lifeexp")
def load_u5mort():         return _load_wb_indicator_wide(WB_DIR / WB_U5MORT_FILE,         "u5mort")
def load_remittances():    return _load_wb_indicator_wide(WB_DIR / WB_REMIT_FILE,           "remit")
def load_resource_rents(): return _load_wb_indicator_wide(WB_DIR / WB_RESOURCE_RENTS_FILE, "res_rents")


def load_rainfall():
    fp = RAIN_DIR / RAINFALL_FILE
    if not fp.exists():
        return pd.DataFrame(columns=["iso", "year", "rainfall_anom"])
    r = _strip_colnames(pd.read_csv(fp))
    r.columns = r.columns.str.lower()
    for alt in ["iso3", "iso3c", "country_code", "countrycode", "code"]:
        if alt in r.columns and "iso" not in r.columns:
            r = r.rename(columns={alt: "iso"}); break
    for alt in ["time", "yr", "date", "period"]:
        if alt in r.columns and "year" not in r.columns:
            r = r.rename(columns={alt: "year"}); break
    if "iso" not in r.columns or "year" not in r.columns:
        return pd.DataFrame(columns=["iso", "year", "rainfall_anom"])
    r["iso"]  = _norm_iso(r["iso"])
    r["year"] = pd.to_numeric(r["year"], errors="coerce")
    r = r.dropna(subset=["year"]).copy()
    r["year"] = r["year"].astype(int)

    anom_cands = ["rainfall_anom", "precip_anom", "rain_anom", "anomaly", "rainfall_anomaly"]
    raw_cands  = ["rainfall", "rainfall_mm", "precip", "precip_mm", "precipitation", "rain", "tp"]
    val_col, is_anom = None, False
    for c in anom_cands:
        if c in r.columns: val_col, is_anom = c, True; break
    if val_col is None:
        for c in raw_cands:
            if c in r.columns: val_col = c; break
    if val_col is None:
        other = [c for c in r.columns if c not in ["iso", "year"]]
        if len(other) == 1: val_col = other[0]
    if val_col is None:
        return pd.DataFrame(columns=["iso", "year", "rainfall_anom"])

    r[val_col] = _to_float(r[val_col])
    if is_anom:
        r["rainfall_anom"] = r[val_col]
    else:
        r = r.sort_values(["iso", "year"]).copy()
        mu = r.groupby("iso")[val_col].transform(
            lambda s: s.expanding(min_periods=3).mean()
        )
        r["rainfall_anom"] = r[val_col] - mu

    if DROP_ISO:   r = r[~r["iso"].isin(DROP_ISO)].copy()
    if DROP_YEARS: r = r[~r["year"].isin(DROP_YEARS)].copy()
    r = _drop_bad_iso_codes(r, "iso")
    r = _add_kosovo_proxy_from_srb(r, "iso")
    out = r[["iso", "year", "rainfall_anom"]].drop_duplicates()
    print(f"✓ Rainfall: {len(out):,} rows")
    return out


# ============================================================
# LOAD EVERYTHING
# ============================================================
print("\n" + "=" * 80)
print(f"Con φ {VERSION}  ~  feature pipeline")
print("LOADING SOURCE DATA")
print("=" * 80)

imf_dict  = load_imf_all()
pip_pct   = load_pip_percentiles()
pip_comp  = load_pip_wide_comparability()
fx        = load_exchange()
rural     = load_rurality()
regions   = load_regions()
edu       = load_ihme_edu()
rain      = load_rainfall()
lifeexp   = load_lifeexp()
u5mort    = load_u5mort()
remit     = load_remittances()
res_rents = load_resource_rents()


# ============================================================
# BUILD ONE RAW MERGED DF FOR AUDIT / WRANGLING
# ============================================================
if AUDIT_VINTAGE not in imf_dict:
    raise ValueError(
        f"AUDIT_VINTAGE={AUDIT_VINTAGE} not found in IMF vintages: {sorted(imf_dict.keys())}"
    )

imf_audit = build_imf_wide_for_vintage(imf_dict, AUDIT_VINTAGE)

keep_isos_audit = set(pip_pct["iso"].unique()) & set(imf_audit["iso"].unique()) - set(DROP_ISO)
if not keep_isos_audit:
    raise ValueError(f"No overlapping ISOs for AUDIT_VINTAGE={AUDIT_VINTAGE}")

source_year_mins = []
for df_ in [imf_audit, pip_pct, pip_comp, fx, rural, edu, rain, lifeexp, u5mort, remit, res_rents]:
    if "year" in df_.columns and len(df_) > 0:
        yy = pd.to_numeric(df_["year"], errors="coerce").dropna()
        if len(yy):
            source_year_mins.append(int(yy.min()))

if not source_year_mins:
    raise ValueError("Could not determine audit panel start year from source files")

audit_start_year          = min(source_year_mins)
audit_end_year_available  = int(pd.to_numeric(imf_audit["year"], errors="coerce").dropna().max())
audit_end_year            = min(AUDIT_VINTAGE + FORECAST_HORIZON_YEARS, audit_end_year_available)

panel_audit = build_iso_year_panel(keep_isos_audit, audit_start_year, audit_end_year)

pip_pct_horizon = pip_pct[
    (pip_pct["iso"].isin(keep_isos_audit)) &
    (pip_pct["year"] >= audit_start_year) &
    (pip_pct["year"] <= audit_end_year)
].copy()

merged_raw_df = expand_outcomes_to_panel(panel_audit, pip_pct_horizon)

for src, src_name in [
    (pip_comp, "pip_comp"), (imf_audit, "imf_audit"), (fx, "fx"),
    (rural, "rural"), (edu, "edu"), (rain, "rain"),
    (lifeexp, "lifeexp"), (u5mort, "u5mort"), (remit, "remit"), (res_rents, "res_rents"),
]:
    filt = (
        src[
            (src["iso"].isin(keep_isos_audit)) &
            (src["year"] >= audit_start_year) &
            (src["year"] <= audit_end_year)
        ] if "year" in src.columns
        else src[src["iso"].isin(keep_isos_audit)]
    )
    merged_raw_df = merge_safe(
        merged_raw_df, filt,
        on=["iso", "year"] if "year" in src.columns else ["iso"],
        how="left",
    )

merged_raw_df = merge_safe(
    merged_raw_df,
    regions[regions["iso"].isin(keep_isos_audit)],
    on=["iso"], how="left",
)


# ============================================================
# BLOCK 3 — BUILD + SAVE FINAL MODEL INPUT FILES
# ============================================================

if RUN_MODE == "single":
    years_to_build = [TEST_YEAR]
else:
    years_to_build = [y for y in range(START_YEAR, END_YEAR + 1) if y in imf_dict]

print("\n" + "=" * 80)
print("BUILDING FINAL MODEL INPUT FILES")
print("=" * 80)
print(f"Con φ version : {VERSION}")
print(f"Output prefix : {OUTPUT_PREFIX}")
print(f"Output dir    : {OUT_DIR}")
print(f"Years to build: {years_to_build}")


def build_final_dataset_for_vintage(
    vintage_year, imf_dict, pip_pct, pip_comp, fx, rural, regions,
    edu, rain, lifeexp, u5mort, remit, res_rents,
):
    """Build a single vintage-controlled Con φ feature file.

    This is where all leakage prevention mechanisms converge:
      1. IMF data comes from the vintage-specific WEO file.
      2. Non-IMF covariates are frozen after vintage_year.
      3. Lagged covariates are frozen after vintage_year.
      4. Region hierarchy is validated for the Bayesian model.
      5. Gov_rev / gov_exp cascade is conditioned on GDP quartile.

    Output feeds both Con φ USE and Con φ WASE.
    """
    print("\n" + "=" * 80)
    print(f"BUILDING Con φ {VERSION} FEATURES  |  IMF vintage {vintage_year}")
    print("=" * 80)

    # ── Load vintage-specific IMF data ────────────────────────
    imf       = build_imf_wide_for_vintage(imf_dict, vintage_year)
    keep_isos = set(pip_pct["iso"].unique()) & set(imf["iso"].unique()) - set(DROP_ISO)
    if not keep_isos:
        raise ValueError(f"No overlapping ISOs for vintage {vintage_year}")

    # ── Determine panel extent ────────────────────────────────
    source_year_mins = []
    for df_ in [imf, pip_pct, pip_comp, fx, rural, edu, rain, lifeexp, u5mort, remit, res_rents]:
        if "year" in df_.columns and len(df_) > 0:
            yy = pd.to_numeric(df_["year"], errors="coerce").dropna()
            if len(yy):
                source_year_mins.append(int(yy.min()))
    panel_start_year         = min(source_year_mins)
    panel_end_year_available = int(pd.to_numeric(imf["year"], errors="coerce").dropna().max())
    panel_end_year           = min(vintage_year + FORECAST_HORIZON_YEARS, panel_end_year_available)
    print(f"Panel years: {panel_start_year}–{panel_end_year}")

    panel = build_iso_year_panel(keep_isos, panel_start_year, panel_end_year)

    pip_pct_horizon = pip_pct[
        (pip_pct["iso"].isin(keep_isos)) &
        (pip_pct["year"] >= panel_start_year) &
        (pip_pct["year"] <= panel_end_year)
    ].copy()

    df = expand_outcomes_to_panel(panel, pip_pct_horizon)

    # ── Merge all source data onto the panel ──────────────────
    for src in [pip_comp, imf, fx, rural, edu, rain, lifeexp, u5mort, remit, res_rents]:
        filt = src[
            (src["iso"].isin(keep_isos)) &
            (src["year"] >= panel_start_year) &
            (src["year"] <= panel_end_year)
        ]
        df = merge_safe(df, filt, on=["iso", "year"], how="left")

    df = merge_safe(df, regions[regions["iso"].isin(keep_isos)], on=["iso"], how="left")

    # ── Post-merge treatment ──────────────────────────────────
    df = df.sort_values(["iso", "year"]).copy()

    # 1a) Interpolate investments/GDP
    if "investments_gdp" in df.columns:
        df["investments_gdp"] = (
            df.groupby("iso")["investments_gdp"]
            .transform(lambda s: s.interpolate(method="linear"))
        )

    # 1b) Smoothed current account (3-year rolling mean, lagged 1 year)
    df["ca_gdp_smooth3"] = (
        df.groupby("iso")["current_account_gdp"]
        .transform(lambda s: s.rolling(3, min_periods=1).mean().shift(1))
    )

    def _freeze_ca(g):
        g      = g.sort_values("year").copy()
        anchor = g.loc[g["year"] == vintage_year, "ca_gdp_smooth3"].dropna()
        if len(anchor) == 0:
            anchor = g.loc[g["year"] <= vintage_year, "ca_gdp_smooth3"].dropna()
        if len(anchor) == 0:
            return g["ca_gdp_smooth3"]
        g.loc[g["year"] > vintage_year, "ca_gdp_smooth3"] = anchor.iloc[-1]
        return g["ca_gdp_smooth3"]

    df["ca_gdp_smooth3"] = df.groupby("iso", group_keys=False).apply(_freeze_ca)
    df["ca_gdp_smooth3"] = df.groupby("iso")["ca_gdp_smooth3"].transform(lambda s: s.bfill())
    for level in ["region23", "region"]:
        if level in df.columns:
            fill = df.groupby(level)["ca_gdp_smooth3"].transform("median")
            df["ca_gdp_smooth3"] = df["ca_gdp_smooth3"].fillna(fill)
    df["ca_gdp_smooth3"] = df["ca_gdp_smooth3"].fillna(df["ca_gdp_smooth3"].median())

    # 1c) Impute remittances (leading zeros → interpolate → ffill)
    df = impute_leading_zero_interp_ffill(df, cols=["remit"])

    # 1d) Impute resource rents (bfill → interpolate → ffill)
    df = impute_bfill_interp_ffill(df, cols=["res_rents"])

    # 1e) Government revenue — GDP-conditioned cascade
    print("  Imputing gov_rev (GDP-conditioned cascade)...")
    if "gov_rev" in df.columns:
        df = cascade_impute_by_gdp_bin(df, "gov_rev", vintage_year)

    # 1f) Government expenditure — GDP-conditioned cascade
    print("  Imputing gov_exp (GDP-conditioned cascade)...")
    if "gov_exp" in df.columns:
        df = cascade_impute_by_gdp_bin(df, "gov_exp", vintage_year)

    # 2) FREEZE non-IMF covariates after vintage year
    df = hold_constant_after_vintage(
        df, cols=POST_VINTAGE_HOLD_CONSTANT_COLS, vintage_year=vintage_year,
    )

    # 3) Create lagged structural covariates AND freeze after vintage
    df = add_lags_and_freeze_after_vintage(
        df,
        cols=["u5mort", "rural_pct", "edu_mean_fem", "remit", "res_rents",
              "gov_rev", "gov_exp"],
        lag=LAG_YEARS, vintage_year=vintage_year,
    )

    # 4) Validate region hierarchy
    df = enforce_nested_regions(df)

    # ── Derived variables ─────────────────────────────────────
    if "gdp_percap_ppp" in df.columns:
        df["log_gdp_pp"] = np.where(
            df["gdp_percap_ppp"] > 0, np.log(df["gdp_percap_ppp"]), np.nan
        )
    else:
        df["log_gdp_pp"] = np.nan

    if "cons" in df.columns:
        df["log_cons"] = np.where(df["cons"] > 0, np.log(df["cons"]), np.nan)
    else:
        df["log_cons"] = np.nan

    if "percentile" in df.columns:
        p = _to_float(df["percentile"]) / 100.0
        p = p.clip(lower=1e-6, upper=1 - 1e-6)
        df["p"]       = p
        df["logit_p"] = np.log(p / (1 - p))
    else:
        df["p"]       = np.nan
        df["logit_p"] = np.nan

    # ── Income group + drop HICs without survey data ──────────
    wb_income = pd.read_csv(WB_DIR / "wb_income_status_2024.csv")
    wb_income = wb_income.rename(
        columns={"Code": "iso", "Income group": "income_group"}
    )[["iso", "income_group"]].copy()
    wb_income["iso"]          = wb_income["iso"].astype(str).str.strip()
    wb_income["income_group"] = wb_income["income_group"].astype(str).str.strip()

    pctiles_keep_isos = set(
        pip_pct.loc[
            (pip_pct["iso"].isin(keep_isos)) & (pip_pct["cons"].notna()), "iso"
        ].astype(str).str.strip().unique()
    )
    df = merge_safe(df, wb_income, on=["iso"], how="left")
    df = df[
        ~(df["income_group"].eq("High income") & ~df["iso"].isin(pctiles_keep_isos))
    ].copy()

    # ── Tidy column ordering ──────────────────────────────────
    first_cols = [
        "iso", "year", "percentile", "pop", "cons", "log_cons", "welfare_type",
        "region", "region23", "income_group",
        "survey_comparability", "comparable_spell_raw",
        "gdp_percap_ppp", "log_gdp_pp", "gdp_percap_lc_current",
        "gdp_percap_lc_real", "gdp_growth",
        "inflation", "cpi", "current_account_gdp", "ca_gdp_smooth3",
        "gov_rev", "gov_rev_lag3",
        "gov_exp", "gov_exp_lag3",
        "fiscal_balance", "debt_gdp",
        "investments_gdp",
        "exchange_rt", "rainfall_anom",
        "remit", "remit_lag3", "res_rents", "res_rents_lag3",
        "rural_pct", "rural_pct_lag3",
        "lifeexp",
        "u5mort", "u5mort_lag3",
        "edu_mean_fem", "edu_mean_fem_lag3", "edu_mean_male",
        "p", "logit_p",
    ]
    first_cols = [c for c in first_cols if c in df.columns]
    other_cols = [c for c in df.columns if c not in first_cols]
    df = (
        df[first_cols + other_cols]
        .sort_values(["iso", "year", "percentile"], na_position="last")
        .reset_index(drop=True)
    )

    # ── Diagnostics ───────────────────────────────────────────
    print(f"✓ Final rows: {len(df):,}")
    print(f"✓ ISOs      : {df['iso'].nunique():,}")
    print(f"✓ Years     : {df['year'].min()}–{df['year'].max()}")
    if "cons" in df.columns:
        print(f"✓ Rows with observed outcome : {df['cons'].notna().sum():,}")
        print(f"✓ Rows without outcome       : {df['cons'].isna().sum():,}")
    for lag_col in [
        "u5mort_lag3", "rural_pct_lag3", "edu_mean_fem_lag3",
        "remit_lag3", "res_rents_lag3", "gov_rev_lag3", "gov_exp_lag3",
    ]:
        if lag_col in df.columns:
            nmiss = df[lag_col].isna().sum()
            print(f"✓ {lag_col}: {df[lag_col].notna().sum():,} non-missing | {nmiss:,} missing")
    for extra_col in ["remit", "res_rents", "gov_rev", "gov_exp", "ca_gdp_smooth3"]:
        if extra_col in df.columns:
            nmiss = df[extra_col].isna().sum()
            print(f"✓ {extra_col}: {df[extra_col].notna().sum():,} non-missing | {nmiss:,} missing")

    n_gdp_growth = df[["iso", "year"]].drop_duplicates().shape[0]
    n_gdp_ok     = df.dropna(subset=["gdp_growth"])[["iso", "year"]].drop_duplicates().shape[0]
    print(
        f"✓ gdp_growth: {n_gdp_ok:,}/{n_gdp_growth:,} country-years "
        f"({100 * n_gdp_ok / max(n_gdp_growth, 1):.1f}%)"
    )

    return df


# ── Build + save all vintage files ────────────────────────────
for yr in years_to_build:
    df_out = build_final_dataset_for_vintage(
        vintage_year=yr, imf_dict=imf_dict, pip_pct=pip_pct, pip_comp=pip_comp,
        fx=fx, rural=rural, regions=regions, edu=edu, rain=rain,
        lifeexp=lifeexp, u5mort=u5mort, remit=remit, res_rents=res_rents,
    )
    out_fp = OUT_DIR / f"{OUTPUT_PREFIX}_{yr}.parquet"
    df_out.to_parquet(out_fp, index=False)
    print(f"✓ Saved: {out_fp.name}")

print("\nDone building Con φ feature files.")


# ============================================================
# BLOCK 4 — MISSINGNESS & VARIABLE AUDIT
# ============================================================

AUDIT_YEAR = AUDIT_VINTAGE
audit_fp   = OUT_DIR / f"{OUTPUT_PREFIX}_{AUDIT_YEAR}.parquet"
df         = pd.read_parquet(audit_fp)
print(f"\nAuditing : {audit_fp.name}")
print(f"Shape    : {df.shape[0]:,} rows × {df.shape[1]} cols")
print(f"ISOs     : {df['iso'].nunique()}  |  Years: {df['year'].min()}–{df['year'].max()}")

ALL_VARS = [
    "cons", "log_cons",
    "gdp_percap_ppp", "log_gdp_pp", "gdp_percap_lc_current",
    "gdp_percap_lc_real", "gdp_growth",
    "inflation", "cpi", "current_account_gdp", "ca_gdp_smooth3",
    "gov_rev", "gov_rev_lag3",
    "gov_exp", "gov_exp_lag3",
    "fiscal_balance", "debt_gdp", "investments_gdp",
    "exchange_rt", "rainfall_anom",
    "remit", "remit_lag3", "res_rents", "res_rents_lag3",
    "rural_pct", "rural_pct_lag3", "lifeexp", "u5mort", "u5mort_lag3",
    "edu_mean_fem", "edu_mean_fem_lag3", "edu_mean_male",
    "region", "region23", "income_group",
    "survey_comparability", "comparable_spell_raw",
]
ALL_VARS = [v for v in ALL_VARS if v in df.columns]

print("\n" + "=" * 70)
print("1. OVERALL COVERAGE (country-year level, deduplicated)")
print("=" * 70)
cy   = df[["iso", "year"]].drop_duplicates()
n_cy = len(cy)
print(f"Total country-years: {n_cy:,}\n")
rows = []
for v in ALL_VARS:
    cy_v = df.dropna(subset=[v])[["iso", "year"]].drop_duplicates()
    n    = len(cy_v)
    rows.append({
        "variable":    v,
        "n_present":   n,
        "n_missing":   n_cy - n,
        "pct_present": round(100 * n / n_cy, 1),
    })
cov = pd.DataFrame(rows).sort_values("pct_present")
print(cov.to_string(index=False))

KEY_VARS = [
    "cons", "gdp_percap_ppp", "gdp_growth", "inflation",
    "current_account_gdp", "ca_gdp_smooth3", "investments_gdp",
    "gov_rev", "gov_rev_lag3",
    "gov_exp", "gov_exp_lag3",
    "exchange_rt", "rainfall_anom",
    "remit", "remit_lag3", "res_rents", "res_rents_lag3",
    "rural_pct", "rural_pct_lag3",
    "lifeexp", "u5mort", "u5mort_lag3",
    "edu_mean_fem", "edu_mean_fem_lag3",
]
KEY_VARS = [v for v in KEY_VARS if v in df.columns]

print("\n" + "=" * 70)
print("2. COVERAGE BY YEAR (n countries with non-missing)")
print("=" * 70)
cy_all   = df[["iso", "year"]].drop_duplicates()
n_by_year = cy_all.groupby("year")["iso"].nunique().rename("n_total")
year_cov  = n_by_year.to_frame()
for v in KEY_VARS:
    cy_v = df.dropna(subset=[v])[["iso", "year"]].drop_duplicates()
    year_cov[v] = cy_v.groupby("year")["iso"].nunique()
year_cov = year_cov.fillna(0).astype(int)
print(year_cov[year_cov.index >= 1990].to_string())

print("\n" + "=" * 70)
print("3. PIP COUNTRIES MISSING EACH COVARIATE ENTIRELY")
print("=" * 70)
pip_isos = set(df.loc[df["cons"].notna(), "iso"].unique())
print(f"ISOs with at least one observed outcome: {len(pip_isos)}\n")
for v in KEY_VARS:
    if v == "cons":
        continue
    v_isos  = set(df.loc[df[v].notna(), "iso"].unique())
    missing = sorted(pip_isos - v_isos)
    if missing:
        print(f"  {v:<26s} ({len(missing):>2d} ISOs): {missing}")
    else:
        print(f"  {v:<26s}  — all PIP countries covered")

SPOT_ISOS = ["TLS", "MNG", "MDA", "THA", "IDN", "KGZ", "SSD", "PSE", "XKX"]
SPOT_ISOS = [i for i in SPOT_ISOS if i in df["iso"].values]
print("\n" + "=" * 70)
print(f"4. SPOT-CHECK: {SPOT_ISOS}")
print("=" * 70)
for iso in SPOT_ISOS:
    sub     = df[df["iso"] == iso][["iso", "year"] + KEY_VARS].drop_duplicates(["iso", "year"])
    n_total = len(sub)
    parts   = []
    for v in KEY_VARS:
        if v in sub.columns:
            n_ok = sub[v].notna().sum()
            if n_ok < n_total:
                parts.append(f"{v}={n_ok}/{n_total}")
    if parts:
        print(f"  {iso}: " + ", ".join(parts))
    else:
        print(f"  {iso}: all key vars fully covered ({n_total} cy)")

NUMERIC_VARS = [v for v in ALL_VARS if v in df.columns and df[v].dtype.kind in "fi"]
print("\n" + "=" * 70)
print("5. DISTRIBUTION SUMMARIES (country-year level)")
print("=" * 70)
cy_df = df[["iso", "year"] + NUMERIC_VARS].drop_duplicates(["iso", "year"])
desc  = cy_df[NUMERIC_VARS].describe().round(2).T
desc["n_missing"] = n_cy - desc["count"].astype(int)
desc  = desc[["count", "n_missing", "mean", "std", "min", "25%", "50%", "75%", "max"]]
print(desc.to_string())

print("\n" + "=" * 70)
print("6. CROSS-VINTAGE CONSISTENCY (iso count & obs outcome rows)")
print("=" * 70)
vintage_summary = []
for yr in range(START_YEAR, END_YEAR + 1):
    fp = OUT_DIR / f"{OUTPUT_PREFIX}_{yr}.parquet"
    if not fp.exists():
        continue
    d = pd.read_parquet(fp, columns=["iso", "year", "cons", "gdp_growth"])
    vintage_summary.append({
        "vintage":        yr,
        "n_isos":         d["iso"].nunique(),
        "year_min":       d["year"].min(),
        "year_max":       d["year"].max(),
        "n_rows":         len(d),
        "n_obs_outcome":  d["cons"].notna().sum(),
        "n_gdp_growth_cy": d.dropna(subset=["gdp_growth"])[["iso", "year"]].drop_duplicates().shape[0],
    })
vs = pd.DataFrame(vintage_summary)
print(vs.to_string(index=False))


# ============================================================
# BLOCK 4b — OUTCOME-CONDITIONAL MISSINGNESS
# ============================================================
# For rows that HAVE an observed outcome (cons not null), which
# covariates are still missing?  This is the operationally
# critical check: any NaN here means the model will either need
# to handle it or the observation is effectively lost.

print("\n" + "=" * 70)
print("7. OUTCOME-CONDITIONAL MISSINGNESS")
print("    (covariate gaps for iso-years WITH observed consumption)")
print("=" * 70)

OUTCOME_COND_VARS = [v for v in KEY_VARS if v != "cons"]

obs_df     = df[df["cons"].notna()].copy()
n_obs_rows = len(obs_df)
n_obs_cy   = obs_df[["iso", "year"]].drop_duplicates().shape[0]
print(f"Rows with observed outcome: {n_obs_rows:,}  ({n_obs_cy:,} country-years)\n")

print("── Per-variable summary ──")
obs_miss_rows = []
for v in OUTCOME_COND_VARS:
    if v not in obs_df.columns:
        continue
    n_na  = obs_df[v].isna().sum()
    cy_na = obs_df.loc[obs_df[v].isna(), ["iso", "year"]].drop_duplicates().shape[0]
    obs_miss_rows.append({
        "variable":     v,
        "rows_missing": n_na,
        "cy_missing":   cy_na,
        "pct_rows_ok":  round(100 * (n_obs_rows - n_na) / max(n_obs_rows, 1), 2),
    })
obs_miss = pd.DataFrame(obs_miss_rows).sort_values("rows_missing", ascending=False)
print(obs_miss.to_string(index=False))

print("\n── Affected iso-years (any covariate missing where outcome exists) ──")
obs_cy  = obs_df[["iso", "year"] + OUTCOME_COND_VARS].drop_duplicates(["iso", "year"])
any_gap = obs_cy[OUTCOME_COND_VARS].isna().any(axis=1)
gap_cy  = obs_cy[any_gap].copy()

if len(gap_cy) == 0:
    print("  None — all outcome iso-years have complete covariates.")
else:
    print(f"  {len(gap_cy):,} country-years with at least one gap:\n")
    gap_detail = []
    for _, row in gap_cy.iterrows():
        missing_vars = [v for v in OUTCOME_COND_VARS if pd.isna(row.get(v))]
        gap_detail.append({
            "iso":          row["iso"],
            "year":         row["year"],
            "n_missing":    len(missing_vars),
            "missing_vars": ", ".join(missing_vars),
        })
    gap_detail_df = pd.DataFrame(gap_detail).sort_values(
        ["n_missing", "iso", "year"], ascending=[False, True, True]
    )
    if len(gap_detail_df) <= 80:
        print(gap_detail_df.to_string(index=False))
    else:
        print(gap_detail_df.head(50).to_string(index=False))
        print(f"\n  ... and {len(gap_detail_df) - 50:,} more rows.")
        iso_gap_counts = gap_detail_df.groupby("iso").size().sort_values(ascending=False)
        print(f"\n  Top ISOs by affected country-years:")
        print(iso_gap_counts.head(20).to_string())


# ============================================================
# BLOCK 5 — VARIABLE NAME SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("8. VARIABLE NAMES IN FINAL DATASET")
print("=" * 70)
print(f"\nTotal columns: {len(df.columns)}\n")

_id_cols       = ["iso", "year", "percentile"]
_outcome_cols  = ["cons", "log_cons", "welfare_type"]
_region_cols   = ["region", "region23", "income_group"]
_survey_cols   = ["survey_comparability", "comparable_spell_raw"]
_gdp_cols      = [c for c in df.columns if "gdp" in c.lower() and c not in _id_cols]
_fiscal_cols   = [
    c for c in df.columns
    if any(k in c for k in ["gov_rev", "gov_exp", "fiscal_balance", "debt_gdp"])
    and c not in _gdp_cols
]
_macro_cols    = [
    c for c in df.columns if c in
    ["inflation", "cpi", "current_account_gdp", "ca_gdp_smooth3",
     "investments_gdp", "exchange_rt"]
]
_structural_cols = [
    c for c in df.columns
    if any(k in c for k in
           ["rural_pct", "edu_mean", "lifeexp", "u5mort", "rainfall_anom",
            "remit", "res_rents", "pop"])
    and c not in _macro_cols
]
_derived_cols  = ["p", "logit_p"]
_all_categorised = set(
    _id_cols + _outcome_cols + _region_cols + _survey_cols +
    _gdp_cols + _fiscal_cols + _macro_cols +
    _structural_cols + _derived_cols
)
_other_cols = [c for c in df.columns if c not in _all_categorised]

for label, cols in [
    ("Identifiers",           _id_cols),
    ("Outcome",               _outcome_cols),
    ("Region / grouping",     _region_cols),
    ("Survey metadata",       _survey_cols),
    ("GDP-related",           _gdp_cols),
    ("Fiscal",                _fiscal_cols),
    ("Other macro",           _macro_cols),
    ("Structural / lagged",   _structural_cols),
    ("Derived (percentile)",  _derived_cols),
    ("Other / uncategorised", _other_cols),
]:
    present = [c for c in cols if c in df.columns]
    if present:
        print(f"  {label}:")
        for c in present:
            dtype = str(df[c].dtype)
            nn    = df[c].notna().sum()
            print(f"    {c:<30s}  {dtype:<10s}  {nn:>12,} non-null")
        print()

print("── Flat column list (copy-paste friendly) ──")
print(f"  {list(df.columns)}")


Con φ v1  ~  feature pipeline
LOADING SOURCE DATA
✓ IMF vintages loaded: [2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
✓ PIP percentiles: 91,872 rows
✓ PIP comparability: 2,309 rows
✓ Exchange rate: 13,350 rows
✓ Rural share: 17,290 rows
✓ Regions: 250 ISOs
✓ IHME education: 12,078 rows
✓ Rainfall: 17,339 rows
✓ lifeexp: 17,556 rows
✓ u5mort: 17,290 rows
✓ remit: 17,556 rows
✓ res_rents: 17,556 rows

BUILDING FINAL MODEL INPUT FILES
Con φ version : v1
Output prefix : conphi_v1_features
Output dir    : /content/drive/MyDrive/conphi/data - model inputs
Years to build: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

BUILDING Con φ v1 FEATURES  |  IMF vintage 2015
Panel years: 1940–2020
  Imputing gov_rev (GDP-conditioned cascade)...
  Imputing gov_exp (GDP-conditioned cascade)...
✓ Final rows: 95,729
✓ ISOs      : 121
✓ Years     : 1940–2020
✓ Rows with observed outcome : 86,724
✓ Rows wi